<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="350px" align="right" border="0"><br>

# Object Oriented Programming

**_Simple Financial Examples_**

Dr. Yves J. Hilpisch | The Python Quants GmbH

<img src="http://hilpisch.com/images/tpq_bootcamp.png" width=350px align=left>

## Simple Short Rate Class

First, some imports.

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8')
%matplotlib inline

Consider the following simple function to calculate a discount factor.

In [ ]:
import numpy as np
def discount_factor(r, t):
    ''' Function to calculate a discount factor.
    
    Parameters
    ==========
    r : float
        positive, constant short rate
    t : float, array of floats
        future date(s), in fraction of years;
        e.g. 0.5 means half a year from now
    
    Returns
    =======
    df : float
        discount factor
    '''
    df = np.exp(-r * t) # use of NumPy universal function for vectorization
    return df

We use the function to plot **discount curves** for different short rates.

In [ ]:
discount_factor(0.05, np.linspace(0, 5))

In [ ]:
plt.figure(figsize=(10, 6))
t = np.linspace(0, 5)
for r in [0.01, 0.05, 0.1]:
    plt.plot(t, discount_factor(r, t), label='r=%4.2f' % r, lw=1.5)
plt.xlabel('years')
plt.ylabel('discount factor')
plt.grid(True)
plt.legend(loc=0);

We now pack that functionality in a class.

In [ ]:
class short_rate(object):
    ''' Class to model a constant short rate object.
    
    Attributes
    ==========
    name: string
        name of the object
    rate: float
        positive, constant short rate
    
    Methods
    =======
    get_rate:
        returns the rate value
    set_rate:
        changes the rate value
    get_discount_factors:
        returns discount factors for given list/array
        of dates/times (as year fractions)
    '''
    def __init__(self, name, rate):
        self.name = name
        self.rate = rate
        
    def get_rate(self):
        ''' Returns the short rate value. '''
        return self.rate
    
    def set_rate(self, rate):
        ''' Changes the short rate value.'''
        self.rate = rate
        
    def get_discount_factors(self, time_list):
        ''' time_list : list/array-like '''
        time_list = np.array(time_list)
        return np.exp(-self.rate * time_list)

The class **in action**.

In [ ]:
sr = short_rate('r', 0.05)

In [ ]:
sr.name, sr.rate

In [ ]:
sr.get_rate()

Let us define a list of future dates in year fractions.

In [ ]:
time_list = [0.0, 0.5, 1.0, 1.25, 1.75, 2.0]  # in year fractions

We can pass this list to our `get_discount_factors` method to get back discount factors.

In [ ]:
sr.get_discount_factors(time_list)

Similarly, we can use the object to plot different discount curves.

In [ ]:
plt.figure(figsize=(10, 6))
for r in [0.025, 0.05, 0.1, 0.15]:
    sr.rate = r
    plt.plot(t, sr.get_discount_factors(t),
             label='r=%4.2f' % sr.rate, lw=1.5)
plt.xlabel('years')
plt.ylabel('discount factor')
plt.grid(True)
plt.legend(loc=0);

Let us now discount **cash flows** with our object.

In [ ]:
sr.set_rate(0.05)
cash_flows = np.array([-100, 50, 75])
time_list = [0.0, 1.0, 2.0]

In [ ]:
disc_facts = sr.get_discount_factors(time_list)

In [ ]:
disc_facts

Combining the single elements yields **present values** and the **net present value**.

In [ ]:
# present values
disc_facts * cash_flows

In [ ]:
# net present value
np.sum(disc_facts * cash_flows)

In [ ]:
sr.set_rate(0.15)
np.sum(sr.get_discount_factors(time_list) * cash_flows)

## Cash Flow Series Class

Next, we define a class to model a **series of cash flows** and to calculate (net) present values.

In [ ]:
class cash_flow_series(object):
    ''' Class to model a cash flows series.
    
    Attributes
    ==========
    name : string
        name of the object
    time_list : list/array-like
        list of (positive) year fractions
    cash_flows : list/array-like
        corresponding list of cash flow values
    short_rate : instance of short_rate class
        short rate object used for discounting
    
    Methods
    =======
    get_present_value_list :
        returns an array with present values
    get_net_present_value :
        returns NPV for cash flow series
    '''
    def __init__(self, name, time_list, cash_flows, short_rate):
        self.name = name
        self.time_list = time_list
        self.cash_flows = cash_flows
        self.short_rate = short_rate
        
    def get_present_value_list(self):
        ''' Returns an array of present values. '''
        df = self.short_rate.get_discount_factors(self.time_list)
        return np.array(self.cash_flows) * df
    
    def get_net_present_value(self):
        ''' Returns the net present value. '''
        return np.sum(self.get_present_value_list())

At instantiation, we pass the `short_rate` object as a parameter.

In [ ]:
sr.set_rate(0.05)
cfs = cash_flow_series('cfs', time_list, cash_flows, sr)

In [ ]:
cfs.cash_flows

In [ ]:
cfs.time_list

In [ ]:
cfs.short_rate

Call the respective **methods**, we get a list of present values and the net present value.

In [ ]:
cfs.get_present_value_list()

In [ ]:
cfs.get_net_present_value()

In [ ]:
sr.set_rate(0.15)

In [ ]:
cfs.get_net_present_value()

Next, we define yet another class which **inherits** from the `cash_flow_series` class.

In [ ]:
class cfs_sensitivity(cash_flow_series):
    def npv_sensitivity(self, short_rates):
        npvs = []
        for rate in short_rates:
            sr.set_rate(rate)  # updates short rate value
            npvs.append(self.get_net_present_value())
        return np.array(npvs)

Instantiation is the same as before.

In [ ]:
cfs_sens = cfs_sensitivity('cfs', time_list, cash_flows, sr)

But we now have an additional method available.

In [ ]:
short_rates = [0.01, 0.025, 0.05, 0.075, 0.1, 0.125, 0.15, 0.2]

In [ ]:
npvs = cfs_sens.npv_sensitivity(short_rates)
npvs

The results **visualized**.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(short_rates, npvs, 'b')
plt.plot(short_rates, npvs, 'ro')
plt.plot((0, max(short_rates)), (0, 0), 'r', lw=2)
plt.grid(True)
plt.xlabel('short rate')
plt.ylabel('net present value');

### Case Study: Implementing a Class for Mean-Variance Portfolio Theory

The class shall:

* receive data for symbols
* calculate log returns
* calculate portfolio return for given weights
* calculate portfolio volatility for given weights
* allow for the adding of symbols ...
* ... and the removal of a symbol
* allow for a Monte Carlo simulation over portfolio weights
* allow for a plotting of the resulting volatility-return combinations

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('http://hilpisch.com/tr_eikon_eod_data.csv',
                                index_col=0, parse_dates=True)

In [ ]:
# data from Thomson Reuters Eikon API
data.info()

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="350px" align="right" border="0"><br>